# (Optional) Colab Setup
If you aren't using Colab, you can delete the following code cell. This is just to help students with mounting to Google Drive to access the other .py files and downloading the data, which is a little trickier on Colab than on your local machine using Jupyter. 

In [1]:
# you will be prompted with a window asking to grant permissions
# from google.colab import drive
# drive.mount("/content/drive")

In [2]:
# fill in the path in your Google Drive in the string below. Note: do not escape slashes or spaces
# import os
# datadir = "/content/assignment1"
# if not os.path.exists(datadir):
#   !ln -s "/content/drive/My Drive/YOUR PATH HERE/assignment1/" $datadir
# os.chdir(datadir)
# !pwd

In [3]:
# downloading Fashion-MNIST
# import os
# os.chdir(os.path.join(datadir,"fashion-mnist/"))
# !chmod +x ./get_data.sh
# !./get_data.sh
# os.chdir(datadir)

# Imports

In [1]:
import random
import numpy as np
from data_process import get_FASHION_data, get_RICE_data
# from scipy.spatial import distance
from models import Perceptron, SVM, Softmax, Logistic
from kaggle_submission import output_submission_csv
%matplotlib inline

# For auto-reloading external modules
# See http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

# Loading Fashion-MNIST

In the following cells we determine the number of images for each split and load the images.
<br /> 
TRAIN_IMAGES + VAL_IMAGES = (0, 60000]
, TEST_IMAGES = 10000

In [2]:
# You can change these numbers for experimentation
# For submission we will use the default values 
TRAIN_IMAGES = 50000
VAL_IMAGES = 10000
normalize = True

In [3]:
data = get_FASHION_data(TRAIN_IMAGES, VAL_IMAGES, normalize=normalize)
X_train_fashion, y_train_fashion = data['X_train'], data['y_train']
X_val_fashion, y_val_fashion = data['X_val'], data['y_val']
X_test_fashion, y_test_fashion = data['X_test'], data['y_test']
n_class_fashion = len(np.unique(y_test_fashion))

# Loading Rice

In [4]:
# loads train / test / val splits of 80%, 20%, 20% 
data = get_RICE_data()
X_train_RICE, y_train_RICE = data['X_train'], data['y_train']
X_val_RICE, y_val_RICE = data['X_val'], data['y_val']
X_test_RICE, y_test_RICE = data['X_test'], data['y_test']
n_class_RICE = len(np.unique(y_test_RICE))

print("Number of train samples: ", X_train_RICE.shape[0])
print("Number of val samples: ", X_val_RICE.shape[0])
print("Number of test samples: ", X_test_RICE.shape[0])

Number of train samples:  10911
Number of val samples:  3637
Number of test samples:  3637


### Get Accuracy

This function computes how well your model performs using accuracy as a metric.

In [5]:
def get_acc(pred, y_test):
    return np.sum(y_test == pred) / len(y_test) * 100

# Perceptron

Perceptron has 2 hyperparameters, "learning rate" and "number of epochs" that you can experiment with:
### Hyperparameter 1: Learning rate
The learning rate controls how much we change the current weights of the classifier during each update. We set it at a default value of 0.5, but you should experiment with different values. Here is a guide to help you find a right learning rate: 
- Try values ranging from 0.0005 to 5.0 to see the impact on model accuracy. 
- If the loss oscillates a lot or the loss diverges, the learning rate may be too high. Try decreasing it by a factor of 10 (e.g. from 0.5 to 0.05). 
- If the loss decreases very slowly, the learning rate may be too low and the training can take a long time. Try increasing it by a factor of 10.
- You can also try adding a learning rate decay to slowly reduce the learning rate over each training epoch. For example, multiply the learning rate by 0.95 after each epoch.
- Plot training and validation accuracy over epochs for different learning rates. This will help you visualize the impact of the learning rate.
- [Here](https://towardsdatascience.com/https-medium-com-dashingaditya-rakhecha-understanding-learning-rate-dd5da26bb6de) is a detailed guide to learning rate.

### Hyperparameter 2: Number of Epochs
- An epoch is a complete iterative pass over the training dataset. During an epoch we predict a label using the classifier and then update the weights of the classifier according to the perceptron update rule for each sample in the training set. 
- By observing the training and validation accuracy graphs, you can determine if the model has converged or not. Try increasing the number of epochs if the model is not converging or has low accuracy. 
- You should try different values for the number of training epochs and report your results.

In this section, you will implement the Perceptron classifier in the **models/perceptron.py** script.

The following code: 
- Creates an instance of the Perceptron classifier class 
- An instance of the Perceptron class is trained on the training data using its train method.
- We use the predict method to find the training accuracy as well as the testing accuracy


## Train Perceptron on Fashion-MNIST

In [16]:
results = {}
np.random.seed(278)


for lr in [0.0005, 0.005, 0.05]:
    for n_epochs in range(10, 31, 10):
        for decay in [0.85, 0.9, 1]:
            percept_fashion = Perceptron(n_class_fashion, lr, n_epochs, decay)
            percept_fashion.train(X_train_fashion, y_train_fashion)
            pred = percept_fashion.predict(X_test_fashion)
            acc = get_acc(pred, y_test_fashion)
            results[lr, n_epochs, decay] = acc
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, decay={decay}: {acc}")

top_3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, decay={params[2]}: accuracy={result}")

print(top_3)
best_params, best_result = top_3[0]

percept_fashion = Perceptron(n_class_fashion, best_params[0], best_params[1], best_params[2])
percept_fashion.train(X_train_fashion, y_train_fashion)

Accuracy for lr=0.0005, n_epochs=10, decay=0.85: 77.64999999999999
Accuracy for lr=0.0005, n_epochs=10, decay=0.9: 77.33
Accuracy for lr=0.0005, n_epochs=10, decay=1: 80.69
Accuracy for lr=0.0005, n_epochs=20, decay=0.85: 79.71000000000001
Accuracy for lr=0.0005, n_epochs=20, decay=0.9: 77.28
Accuracy for lr=0.0005, n_epochs=20, decay=1: 80.96
Accuracy for lr=0.0005, n_epochs=30, decay=0.85: 79.49000000000001
Accuracy for lr=0.0005, n_epochs=30, decay=0.9: 78.38000000000001
Accuracy for lr=0.0005, n_epochs=30, decay=1: 77.85
Accuracy for lr=0.005, n_epochs=10, decay=0.85: 79.17999999999999
Accuracy for lr=0.005, n_epochs=10, decay=0.9: 77.69
Accuracy for lr=0.005, n_epochs=10, decay=1: 75.29
Accuracy for lr=0.005, n_epochs=20, decay=0.85: 79.32000000000001
Accuracy for lr=0.005, n_epochs=20, decay=0.9: 76.53
Accuracy for lr=0.005, n_epochs=20, decay=1: 77.7
Accuracy for lr=0.005, n_epochs=30, decay=0.85: 79.79
Accuracy for lr=0.005, n_epochs=30, decay=0.9: 79.75
Accuracy for lr=0.005, 

In [ ]:
# Best achieved accuracy: 
l_rate = 0.0005
n_epochs = 20
decay = 1

# From the sample parameters:
l_rate = 0.5
n_epochs = 20
decay = 1

# ercept_fashion = Perceptron(n_class_fashion, l_rate, n_epochs, decay)
# percept_fashion.train(X_train_fashion, y_train_fashion)
# pred = percept_fashion.predict(X_test_fashion)
# acc = get_acc(pred, y_test_fashion)

percept_fashion = Perceptron(n_class_fashion, l_rate, n_epochs, decay)
percept_fashion.train(X_train_fashion, y_train_fashion)



In [31]:
pred_percept = percept_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_percept, y_train_fashion)))

The training accuracy is given by: 82.984000


### Validate Perceptron on Fashion-MNIST

In [32]:
pred_percept = percept_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_percept, y_val_fashion)))

The validation accuracy is given by: 80.950000


### Test Perceptron on Fashion-MNIST

In [33]:
pred_percept = percept_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_percept, y_test_fashion)))

The testing accuracy is given by: 79.990000


### Perceptron_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy, output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [13]:
output_submission_csv('kaggle/perceptron_submission_fashion.csv', percept_fashion.predict(X_test_fashion))

## Train Perceptron on Rice

In [ ]:
np.random.seed(278)

results = {}
for lr in [0.0005, 0.005, 0.05, 0.1, 0.2, 0.5, 0.7, 1.0, 2.0, 5.0]:
    for n_epochs in range(30, 101, 10):
        for decay in [0.85, 0.9, 1]:
            percept_RICE = Perceptron(n_class_RICE, lr, n_epochs, decay)
            percept_RICE.train(X_train_RICE, y_train_RICE)
            pred = percept_RICE.predict(X_test_RICE)
            acc = get_acc(pred, y_test_RICE)
            results[lr, n_epochs, decay] = acc
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, decay={decay}: {acc}")


top_3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, decay={params[2]}: accuracy={result}")

print(top_3)
best_params, best_result = top_3[0]

percept_fashion = Perceptron(n_class_fashion, best_params[0], best_params[1], best_params[2])
percept_fashion.train(X_train_fashion, y_train_fashion)


Accuracy for lr=0.0005, n_epochs=30, decay=0.85: 99.83502886994776
Accuracy for lr=0.0005, n_epochs=30, decay=0.9: 99.86252405828981
Accuracy for lr=0.0005, n_epochs=30, decay=1: 99.7250481165796
Accuracy for lr=0.0005, n_epochs=40, decay=0.85: 99.91751443497388
Accuracy for lr=0.0005, n_epochs=40, decay=0.9: 99.86252405828981
Accuracy for lr=0.0005, n_epochs=40, decay=1: 89.90926587847127
Accuracy for lr=0.0005, n_epochs=50, decay=0.85: 99.86252405828981
Accuracy for lr=0.0005, n_epochs=50, decay=0.9: 99.86252405828981
Accuracy for lr=0.0005, n_epochs=50, decay=1: 99.0376684080286
Accuracy for lr=0.0005, n_epochs=60, decay=0.85: 99.89001924663185
Accuracy for lr=0.0005, n_epochs=60, decay=0.9: 99.91751443497388
Accuracy for lr=0.0005, n_epochs=60, decay=1: 99.61506736321144
Accuracy for lr=0.0005, n_epochs=70, decay=0.85: 99.89001924663185
Accuracy for lr=0.0005, n_epochs=70, decay=0.9: 99.89001924663185
Accuracy for lr=0.0005, n_epochs=70, decay=1: 98.70772614792412
Accuracy for lr=0

/Users/edu/Documents/UIUC/CS444/assignment1/models/perceptron.py:59: RuntimeWarning: overflow encountered in add
  np.transpose(self.w)[current_class] += (self.lr * self.reg_const / samples) * np.transpose(self.w)[current_class]


Accuracy for lr=1.0, n_epochs=90, decay=1: 45.94445971954908
Accuracy for lr=1.0, n_epochs=100, decay=0.85: 99.78003849326367
Accuracy for lr=1.0, n_epochs=100, decay=0.9: 99.80753368160572
Accuracy for lr=1.0, n_epochs=100, decay=1: 45.94445971954908
Accuracy for lr=2.0, n_epochs=30, decay=0.85: 99.80753368160572
Accuracy for lr=2.0, n_epochs=30, decay=0.9: 99.75254330492163
Accuracy for lr=2.0, n_epochs=30, decay=1: 99.39510585647511
Accuracy for lr=2.0, n_epochs=40, decay=0.85: 99.56007698652736
Accuracy for lr=2.0, n_epochs=40, decay=0.9: 98.95518284300248
Accuracy for lr=2.0, n_epochs=40, decay=1: 99.5875721748694
Accuracy for lr=2.0, n_epochs=50, decay=0.85: 99.67005773989553
Accuracy for lr=2.0, n_epochs=50, decay=0.9: 99.61506736321144
Accuracy for lr=2.0, n_epochs=50, decay=1: 45.94445971954908
Accuracy for lr=2.0, n_epochs=60, decay=0.85: 99.53258179818532
Accuracy for lr=2.0, n_epochs=60, decay=0.9: 99.53258179818532
Accuracy for lr=2.0, n_epochs=60, decay=1: 45.944459719549

In [49]:
# Best achieved accuracy:
l_rate = 0.005
n_epochs = 100
decay = 1

# From the sample parameters:
# l_rate = 0.5
# n_epochs = 10


percept_RICE = Perceptron(n_class_RICE, l_rate, n_epochs, decay)
percept_RICE.train(X_train_RICE, y_train_RICE)

pred_percept = percept_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_percept, y_train_RICE)))

Epoch 0 Accuracy 45.5228668316378
Epoch 1 Accuracy 79.51608468518009
Epoch 2 Accuracy 81.46824305746495
Epoch 3 Accuracy 78.59041334433141
Epoch 4 Accuracy 94.99587572174869
Epoch 5 Accuracy 95.7290807442031
Epoch 6 Accuracy 98.79937677573092
Epoch 7 Accuracy 96.78306296398131
Epoch 8 Accuracy 98.46026945284575
Epoch 9 Accuracy 99.80753368160572
Epoch 10 Accuracy 98.92768765466043
Epoch 11 Accuracy 99.86252405828981
Epoch 12 Accuracy 99.43176610759784
Epoch 13 Accuracy 99.4500962331592
Epoch 14 Accuracy 99.83502886994776
Epoch 15 Accuracy 99.51425167262396
Epoch 16 Accuracy 99.86252405828981
Epoch 17 Accuracy 99.89001924663185
Epoch 18 Accuracy 99.541746860966
Epoch 19 Accuracy 99.6792228026762
Epoch 20 Accuracy 99.16597928695812
Epoch 21 Accuracy 99.3584456053524
Epoch 22 Accuracy 99.57840711208871
Epoch 23 Accuracy 99.541746860966
Epoch 24 Accuracy 98.79021171295024
Epoch 25 Accuracy 99.2209696636422
Epoch 26 Accuracy 98.50609476674916
Epoch 27 Accuracy 99.55091192374668
Epoch 28 Acc

### Validate Perceptron on Rice

In [50]:
pred_percept = percept_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_percept, y_val_RICE)))

The validation accuracy is given by: 99.862524


### Test Perceptron on Rice

In [51]:
pred_percept = percept_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_percept, y_test_RICE)))

The testing accuracy is given by: 99.890019


# Support Vector Machines (with SGD)

Next, you will implement a "soft margin" SVM. In this formulation you will maximize the margin between positive and negative training examples and penalize margin violations using a hinge loss.

We will optimize the SVM loss using SGD. This means you must compute the loss function with respect to model weights. You will use this gradient to update the model weights.

SVM optimized with SGD has 3 hyperparameters that you can experiment with:
- **Learning rate** - similar to as defined above in Perceptron, this parameter scales by how much the weights are changed according to the calculated gradient update. 
- **Epochs** - similar to as defined above in Perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case it is a coefficient on the term which maximizes the margin. You could try different values. The default value is set to 0.05.

You will implement the SVM using SGD in the **models/svm.py**

The following code: 
- Creates an instance of the SVM classifier class 
- The train function of the SVM class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

## Train SVM on Fashion-MNIST

In [211]:
np.random.seed(278)

accumulator = 0
lr_ranges = [0.05, 0.5, 5.0]
n_epochs_ranges = [10, 20]
reg_const_ranges = [0.005, 0.05, 0.5]
decay_ranges = [0.85, 0.9, 1]
total = len(lr_ranges) * len(n_epochs_ranges) * len(reg_const_ranges) * len(decay_ranges)

results = {}
for lr in lr_ranges:
    for n_epochs in n_epochs_ranges:
        for reg_const in reg_const_ranges:
            for decay in decay_ranges:
                svm_fashion = SVM(n_class_fashion, lr, n_epochs, reg_const, decay)
                svm_fashion.train(X_train_fashion, y_train_fashion)
                pred = svm_fashion.predict(X_test_fashion)
                acc = get_acc(pred, y_test_fashion)
                results[lr, n_epochs, reg_const, decay] = acc
                accumulator += 1
                print(f"\r{accumulator}/{total}", end="")
                # print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}: {acc}")

top_3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[3]}: accuracy={result}")

print(top_3)
best_params, best_result = top_3[0]

svm_fashion = SVM(n_class_fashion, best_params[0], best_params[1], best_params[2], best_params[3])
svm_fashion.train(X_train_fashion, y_train_fashion)

54/54Top 3 hyperparameter combinations:
lr=5.0, n_epochs=20, reg_const=0.005, decay=1: accuracy=82.65
lr=0.5, n_epochs=10, reg_const=0.05, decay=0.85: accuracy=82.33
lr=0.5, n_epochs=20, reg_const=0.5, decay=0.85: accuracy=82.28999999999999
[((5.0, 20, 0.005, 1), np.float64(82.65)), ((0.5, 10, 0.05, 0.85), np.float64(82.33)), ((0.5, 20, 0.5, 0.85), np.float64(82.28999999999999))]


In [219]:
np.random.seed(278)

# Adjust the following three hyperparameters to achieve the best accuracy:
lr = 0.5
n_epochs = 10
reg_const = 0.05
decay_rate = 0.85

svm_fashion = SVM(n_class_fashion, lr, n_epochs, reg_const, decay_rate)

# Refer to hints in the calculate_gradient and train methods of the SVM class in "models/svm.py"
svm_fashion.train(X_train_fashion, y_train_fashion)

In [220]:
# Best achieved accuracy:
# l_rate = 0.5
# n_epochs = 70
# reg_const = 0.005

# svm_fashion = SVM(n_class_fashion, l_rate, n_epochs, reg_const)
# svm_fashion.train(X_train_fashion, y_train_fashion)

pred_svm = svm_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_svm, y_train_fashion)))

The training accuracy is given by: 84.326000


### Validate SVM on Fashion-MNIST

In [221]:
pred_svm = svm_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_svm, y_val_fashion)))

The validation accuracy is given by: 83.270000


### Test SVM on Fashion-MNIST

In [222]:
pred_svm = svm_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_svm, y_test_fashion)))

The testing accuracy is given by: 82.370000


### SVM_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [206]:
output_submission_csv('kaggle/svm_submission_fashion.csv', svm_fashion.predict(X_test_fashion))

## Train SVM on Rice

In [190]:
# Adjust the following three hyperparameters to achieve the best accuracy: 
np.random.seed(278)

results = {}
for lr in [0.0005, 0.005, 0.05, 0.1, 1, 5.0]:
    for n_epochs in range(30, 51, 10):
        for reg_const in [0.005, 0.01, 0.05, 0.5]:
            for decay in [0.85, 0.95, 1]:
                svm_RICE = SVM(n_class_RICE, lr, n_epochs, reg_const)
                svm_RICE.train(X_train_RICE, y_train_RICE)
                pred = svm_RICE.predict(X_test_RICE)
                acc = get_acc(pred, y_test_RICE)
                results[lr, n_epochs, reg_const, decay] = acc
                print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}, decay={decay} : {acc}")

top_3 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[2]}: accuracy={result}")
print(top_3)
best_params, best_result = top_3[0]

svm_RICE = SVM(n_class_RICE, best_params[0], best_params[1], best_params[2], best_params[3])
svm_RICE.train(X_train_RICE, y_train_RICE)

Accuracy for lr=0.0005, n_epochs=30, reg_const=0.005, decay=0.85 : 54.05554028045092
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.005, decay=0.95 : 66.2359087159747
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.005, decay=1 : 54.05554028045092
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.01, decay=0.85 : 48.47401704701677
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.01, decay=0.95 : 54.05554028045092
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.01, decay=1 : 45.94445971954908
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.05, decay=0.85 : 78.03134451470993
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.05, decay=0.95 : 54.05554028045092
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.05, decay=1 : 54.05554028045092
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.5, decay=0.85 : 71.95490789111906
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.5, decay=0.95 : 45.94445971954908
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.5, decay=1 : 49.76629089909266
Accur

In [151]:
np.random.seed(278)

# Adjust the following three hyperparameters to achieve the best accuracy: 
lr = 0.05
n_epochs = 50
reg_const = 0.01
decay = 0.95

svm_RICE = SVM(n_class_RICE, lr, n_epochs, reg_const, decay)
svm_RICE.train(X_train_RICE, y_train_RICE)

In [152]:
# These are the best values from the previous cell
# lr = 2.5
# n_epochs = 80
# reg_const = 0.01


# svm_RICE = SVM(n_class_RICE, lr, n_epochs, reg_const)
# svm_RICE.train(X_train_RICE, y_train_RICE)

pred_svm = svm_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_svm, y_train_RICE)))

The training accuracy is given by: 98.817707


### Validate SVM on Rice

In [153]:
pred_svm = svm_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_svm, y_val_RICE)))

The validation accuracy is given by: 98.872697


## Test SVM on Rice

In [154]:
pred_svm = svm_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_svm, y_test_RICE)))

The testing accuracy is given by: 98.735221


# Softmax Classifier (with SGD)

Next, you will train a Softmax classifier. This classifier consists of a linear function of the input data followed by a softmax function which outputs a vector of dimension C (number of classes) for each data point. Each entry of the softmax output vector corresponds to a confidence in one of the C classes, and like a probability distribution, the entries of the output vector sum to 1. We use a cross-entropy loss on this sotmax output to train the model. 

Check the following link as an additional resource on softmax classification: http://cs231n.github.io/linear-classify/#softmax

Once again we will train the classifier with SGD. This means you need to compute the gradients of the softmax cross-entropy loss function according to the weights and update the weights using this gradient. Check the following link to help with implementing the gradient updates: https://deepnotes.io/softmax-crossentropy

The softmax classifier has 3 hyperparameters that you can experiment with:
- **Learning rate** - As above, this controls how much the model weights are updated with respect to their gradient.
- **Number of Epochs** - As described for perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case, we minimize the L2 norm of the model weights as regularization, so the regularization constant is a coefficient on the L2 norm in the combined cross-entropy and regularization objective.

You will implement a softmax classifier using SGD in the **models/softmax.py**

The following code: 
- Creates an instance of the Softmax classifier class 
- The train function of the Softmax class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

## Train Softmax on Fashion-MNIST

In [ ]:
# lr = 1
# n_epochs = 50
# reg_const = 0.05

lr_range = [0.0005, 0.005, 0.01, 0.05, 0.1, 0.5]
n_epochs_range = range(50, 101, 10)
reg_const_range = [0.005, 0.05, 0.5]
decay_range =  [0.85, 0.9, 0.95, 1]
batch_size_range = [1024, 2048, 4096]

accumulator = 0
total = len(lr_range) * len(n_epochs_range) * len(reg_const_range) * len(decay_range) * len(batch_size_range)

results = {}
for lr in lr_range:
    for n_epochs in n_epochs_range:
        for reg_const in reg_const_range:
            for decay in decay_range:
                for batch_size in batch_size_range:
                    softmax_fashion = Softmax(n_class_fashion, lr, n_epochs, reg_const, decay, batch_size)
                    softmax_fashion.train(X_train_fashion, y_train_fashion)
                    pred = softmax_fashion.predict(X_test_fashion)
                    acc = get_acc(pred, y_test_fashion)
                    results[lr, n_epochs, reg_const, decay, batch_size] = acc
                    accumulator += 1
                    print(f"\r{accumulator}/{total}", end="")
                    # print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}, decay={decay}: {acc}")

top_5 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 hyperparameter combinations:")
for params, result in top_5:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[3]}: accuracy={result}")

print(top_5)
best_params, best_result = top_5[0]
softmax_fashion = Softmax(n_class_fashion, best_params[0], best_params[1], best_params[2], best_params[3], best_params[4])

# Refer to hints in the calculate_gradient and train methods of the Softmax class in "models/softmax.py"
softmax_fashion.train(X_train_fashion, y_train_fashion)

1296/1296Top 5 hyperparameter combinations:
lr=0.5, n_epochs=70, reg_const=0.05, decay=0.85: accuracy=82.88
lr=0.5, n_epochs=90, reg_const=0.005, decay=0.9: accuracy=82.83
lr=0.5, n_epochs=90, reg_const=0.5, decay=0.85: accuracy=82.8
lr=0.1, n_epochs=70, reg_const=0.5, decay=0.85: accuracy=82.78
lr=0.5, n_epochs=50, reg_const=0.5, decay=0.9: accuracy=82.78
[((0.5, 70, 0.05, 0.85), np.float64(82.88)), ((0.5, 90, 0.005, 0.9), np.float64(82.83)), ((0.5, 90, 0.5, 0.85), np.float64(82.8)), ((0.1, 70, 0.5, 0.85), np.float64(82.78)), ((0.5, 50, 0.5, 0.9), np.float64(82.78))]


In [185]:
# Best parameters:
l_rate = 0.5
n_epochs = 70
reg_const = 0.05
decay = 0.85
np.random.seed(278)

# l_rate = 0.05
# n_epochs = 90
# reg_const = 0.005
# decay = False

for bs in batch_size_range:
    softmax_fashion = Softmax(n_class_fashion, l_rate, n_epochs, reg_const, decay, bs)
    softmax_fashion.train(X_train_fashion, y_train_fashion)

    print(f"Batch size: {bs}")
    pred_softmax = softmax_fashion.predict(X_train_fashion)
    print('The training accuracy is given by: %f' % (get_acc(pred_softmax, y_train_fashion)))
    pred_softmax = softmax_fashion.predict(X_val_fashion)
    print('The validation accuracy is given by: %f' % (get_acc(pred_softmax, y_val_fashion)))
    pred_softmax = softmax_fashion.predict(X_test_fashion)
    print('The testing accuracy is given by: %f' % (get_acc(pred_softmax, y_test_fashion)))
    print()

Batch size: 1024
The training accuracy is given by: 85.714000
The validation accuracy is given by: 82.320000
The testing accuracy is given by: 81.610000

Batch size: 2048
The training accuracy is given by: 85.530000
The validation accuracy is given by: 82.990000
The testing accuracy is given by: 82.350000

Batch size: 4096
The training accuracy is given by: 85.024000
The validation accuracy is given by: 83.750000
The testing accuracy is given by: 82.720000



In [ ]:
l_rate = 0.5
n_epochs = 70
reg_const = 0.05
decay = 0.85
bs = 4096
np.random.seed(278)

softmax_fashion = Softmax(n_class_fashion, l_rate, n_epochs, reg_const, decay, bs)
softmax_fashion.train(X_train_fashion, y_train_fashion)

In [180]:
pred_softmax = softmax_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_softmax, y_train_fashion)))

The training accuracy is given by: 85.076000


### Validate Softmax on Fashion-MNIST

In [181]:
pred_softmax = softmax_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_softmax, y_val_fashion)))

The validation accuracy is given by: 83.630000


### Testing Softmax on Fashion-MNIST

In [182]:
pred_softmax = softmax_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_softmax, y_test_fashion)))

The testing accuracy is given by: 82.790000


### Softmax_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [183]:
output_submission_csv('kaggle/softmax_submission_fashion.csv', softmax_fashion.predict(X_test_fashion))

## Train Softmax on Rice

In [ ]:
# lr = 0.5
# n_epochs = 10
# reg_const = 0.05

lr_range = [0.0005, 0.005, 0.05, 0.5]
n_epochs_range = range(100, 201, 10)
reg_const_range = [0.005, 0.05, 0.5, 1, 2.5, 5]
decay_range =  [0.85, 0.9, 1]
batch_size_range = [1024, 2048, 4096]


accumulator = 0
total = len(lr_range) * len(n_epochs_range) * len(reg_const_range) * len(decay_range) * len(batch_size_range)

results = {}
for lr in lr_range:
    for n_epochs in n_epochs_range:
        for reg_const in reg_const_range:
            for decay in decay_range:
                for batch_size in batch_size_range:
                    softmax_RICE = Softmax(n_class_RICE, lr, n_epochs, reg_const, decay, batch_size)
                    softmax_RICE.train(X_train_RICE, y_train_RICE)
                    pred = softmax_RICE.predict(X_test_RICE)
                    acc = get_acc(pred, y_test_RICE)
                    results[lr, n_epochs, reg_const, decay, batch_size] = acc
                    accumulator += 1
                    print(f"\r{accumulator}/{total}", end="")
                    # print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}, decay={decay}: {acc}")

top_5 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 hyperparameter combinations:")
for params, result in top_5:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[3]}: accuracy={result}")

print(top_5)
best_params, best_result = top_5[0]

softmax_RICE = Softmax(n_class_RICE, best_params[0], best_params[1], best_params[2], best_params[3], best_params[4])
softmax_RICE.train(X_train_RICE, y_train_RICE)

2265/2376

/Users/edu/Documents/UIUC/CS444/assignment1/models/softmax.py:42: RuntimeWarning: overflow encountered in subtract
  feature_scores = np.exp(fs - np.max(fs, axis=1, keepdims=True))
/Users/edu/Documents/UIUC/CS444/assignment1/models/softmax.py:42: RuntimeWarning: invalid value encountered in subtract
  feature_scores = np.exp(fs - np.max(fs, axis=1, keepdims=True))


2376/2376Top 5 hyperparameter combinations:
lr=0.0005, n_epochs=110, reg_const=0.05, decay=0.9: accuracy=99.67005773989553
lr=0.0005, n_epochs=160, reg_const=2.5, decay=0.9: accuracy=99.64256255155348
lr=0.0005, n_epochs=180, reg_const=0.5, decay=0.9: accuracy=99.64256255155348
lr=0.0005, n_epochs=200, reg_const=0.005, decay=0.85: accuracy=99.64256255155348
lr=0.0005, n_epochs=130, reg_const=0.5, decay=0.9: accuracy=99.61506736321144
[((0.0005, 110, 0.05, 0.9, 1024), np.float64(99.67005773989553)), ((0.0005, 160, 2.5, 0.9, 1024), np.float64(99.64256255155348)), ((0.0005, 180, 0.5, 0.9, 1024), np.float64(99.64256255155348)), ((0.0005, 200, 0.005, 0.85, 1024), np.float64(99.64256255155348)), ((0.0005, 130, 0.5, 0.9, 1024), np.float64(99.61506736321144))]


In [187]:
# Best parameters:
l_rate = 0.0005
n_epochs = 110
reg_const = 0.05
decay = 0.9
batch_size = 1024
np.random.seed(278)

softmax_RICE = Softmax(n_class_RICE, l_rate, n_epochs, reg_const, decay, batch_size)
softmax_RICE.train(X_train_RICE, y_train_RICE)

pred_softmax = softmax_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_softmax, y_train_RICE)))

The training accuracy is given by: 99.120154


### Validate Softmax on Rice

In [188]:
pred_softmax = softmax_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_softmax, y_val_RICE)))

The validation accuracy is given by: 99.065164


### Testing Softmax on Rice

In [189]:
pred_softmax = softmax_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_softmax, y_test_RICE)))

The testing accuracy is given by: 99.120154


# Logistic Classifier

The Logistic Classifier has 2 hyperparameters that you can experiment with:
- **Learning rate** - similar to as defined above in Perceptron, this parameter scales by how much the weights are changed according to the calculated gradient update. 
- **Number of Epochs** - As described for perceptron.
- **Threshold** - The decision boundary of the classifier.


You will implement the Logistic Classifier in the **models/logistic.py**

The following code: 
- Creates an instance of the Logistic classifier class 
- The train function of the Logistic class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

### Training Logistic Classifer

In [ ]:
learning_rate = 0.5
n_epochs = 10
threshold = 0.5

lr = Logistic(learning_rate, n_epochs, threshold)

# Refer to hints in the sigmoid and train methods of the Logistic class in "models/logistic.py"
lr.train(X_train_RICE, y_train_RICE)

In [ ]:
pred_lr = lr.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_lr, y_train_RICE)))

### Validate Logistic Classifer

In [ ]:
pred_lr = lr.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_lr, y_val_RICE)))

### Test Logistic Classifier

In [ ]:
pred_lr = lr.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_lr, y_test_RICE)))